# 02 — Ablation Runs
Three-way comparison to isolate the value of event data:

| Run | Model | Events | Checkpoint |
|-----|-------|--------|------------|
| `baseline` | RGBBaselineUNet (17M) | None | `vimeo_v6_baseline` |
| `dual_zero` | DualEncoderUNet (28M) | All zeros | `vimeo_v6_dual_zero` |
| `dual_events` | DualEncoderUNet (28M) | Real v2e | `vimeo_v6_dual` (already done: **28.60 dB**) |

**Change `RUN_MODE` below and run all cells.** Data extraction is shared.

In [ ]:
# ── Cell 1: Setup & Mount ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/493Project')

%pip install -q torchmetrics

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 25.0 MB/s eta 0:00:00


In [ ]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import importlib
import src.models, src.train, src.losses, src.data
importlib.reload(src.models)
importlib.reload(src.losses)
importlib.reload(src.train)
importlib.reload(src.data)

import os
import glob
import random
import torch
from torch.amp import GradScaler
from torch.utils.data import DataLoader

from src.data import VimeoTripletDataset
from src.train import (
    build_model,
    build_criterion,
    build_optimizer,
    build_scheduler,
    load_checkpoint,
    run_training,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────────────────
# >>> CHANGE THIS to select which run <<<
RUN_MODE = 'dual_zero'   # 'baseline' | 'dual_zero'

# Shared config (same as the dual_events run that got 28.60 dB)
shared = dict(
    vimeo_root     = '/content/drive/MyDrive/493Project/data/vimeo',
    local_root     = '/content/local_vimeo',
    patch_size     = 256,
    batch_size     = 16,
    num_workers    = 4,
    lr             = 2e-4,
    lr_min         = 1e-6,
    weight_decay   = 0.0,
    warmup_epochs  = 0,
    lambda_char            = 1.0,
    lambda_perceptual      = 0.0,
    lambda_ssim            = 0.0,
    lambda_event_weighted  = 0.0,
    num_epochs     = 40,
    max_norm       = 1.0,
)

configs = {
    'baseline': dict(
        **shared,
        model_type     = 'baseline',
        zero_events    = False,
        checkpoint_dir = '/content/drive/MyDrive/493Project/checkpoints/vimeo_v6_baseline',
    ),
    'dual_zero': dict(
        **shared,
        model_type     = 'dual',
        zero_events    = True,
        checkpoint_dir = '/content/drive/MyDrive/493Project/checkpoints/vimeo_v6_dual_zero',
    ),
}

cfg = configs[RUN_MODE]
os.makedirs(cfg['checkpoint_dir'], exist_ok=True)
print(f'Run mode: {RUN_MODE}')
print(f'Model: {cfg["model_type"]} | Zero events: {cfg.get("zero_events", False)}')
print(f'Checkpoint: {cfg["checkpoint_dir"]}')

Run mode: dual_zero
Model: dual | Zero events: True
Checkpoint: /content/drive/MyDrive/493Project/checkpoints/vimeo_v6_dual_zero


In [ ]:
# ── Cell 4: Load data — extract tar to local SSD ────────────────────────────
import subprocess

vimeo_root = cfg['vimeo_root']
local_root = cfg['local_root']
os.makedirs(local_root, exist_ok=True)

tar_on_drive = os.path.join(vimeo_root, 'processed.tar')
marker = os.path.join(local_root, '.copy_done')

if not os.path.exists(marker):
    assert os.path.exists(tar_on_drive), \
        f'processed.tar not found on Drive — run 00b_tar_processed.ipynb first'

    subprocess.run(['apt-get', 'install', '-qq', '-y', 'pv'], capture_output=True)
    total_bytes = os.path.getsize(tar_on_drive)
    print(f'Extracting processed.tar ({total_bytes / 1e9:.1f} GB) to local SSD...')
    subprocess.run(
        f'pv -f -s {total_bytes} "{tar_on_drive}" | tar xf - -C "{local_root}"',
        shell=True, check=True,
    )
    open(marker, 'w').close()
    print('Done.')
else:
    print('Local data already exists, skipping extraction.')

def load_split(split_file):
    path = os.path.join(vimeo_root, split_file)
    with open(path) as f:
        entries = [line.strip() for line in f if line.strip()]
    dirs = []
    for rel in entries:
        d = os.path.join(local_root, rel)
        if os.path.isdir(d):
            dirs.append((rel, d))
    return dirs

train_entries = load_split('tri_trainlist.txt')
val_entries   = load_split('tri_vallist.txt')

if len(val_entries) == 0 and len(train_entries) > 0:
    print(f'No processed val triplets found — splitting 90/10 from {len(train_entries)} train entries')
    rng = random.Random(42)
    all_entries = list(train_entries)
    rng.shuffle(all_entries)
    n_val = max(1, int(len(all_entries) * 0.10))
    val_entries   = all_entries[:n_val]
    train_entries = all_entries[n_val:]

train_dirs = [d for _, d in train_entries]
val_dirs   = [d for _, d in val_entries]
print(f'Train: {len(train_dirs)} | Val: {len(val_dirs)}')

Extracting processed.tar (5.3 GB) to local SSD...
Done.
No processed val triplets found — splitting 90/10 from 1900 train entries
Train: 1710 | Val: 190


In [ ]:
# ── Cell 5: Create datasets and dataloaders ──────────────────────────────────
zero_events = cfg.get('zero_events', False)

train_dataset = VimeoTripletDataset(train_dirs, patch_size=cfg['patch_size'],
                                     augment=True, zero_events=zero_events)
val_dataset   = VimeoTripletDataset(val_dirs,   patch_size=cfg['patch_size'],
                                     augment=False, zero_events=zero_events)

train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'], shuffle=True,
                          num_workers=cfg['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg['batch_size'], shuffle=False,
                          num_workers=cfg['num_workers'], pin_memory=True)

print(f'Train: {len(train_dataset)} clips, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} clips, {len(val_loader)} batches')
print(f'Zero events: {zero_events}')

Train: 1710 clips, 107 batches
Val:   190 clips, 12 batches
Zero events: True


In [ ]:
# ── Cell 6: Model, optimizer, scheduler, loss ────────────────────────────────
model     = build_model(cfg, device)
optimizer = build_optimizer(model, cfg)
scheduler = build_scheduler(optimizer, cfg)
scaler    = GradScaler('cuda', enabled=False)
criterion = build_criterion(cfg, device)

start_epoch, best_psnr = load_checkpoint(
    cfg['checkpoint_dir'], model, optimizer, scheduler, device
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {cfg["model_type"]} | Parameters: {n_params:,}')
print(f'LR: {cfg["lr"]} | AMP: {scaler.is_enabled()}')
print(f'Loss: Charb({cfg["lambda_char"]}) + Perc({cfg["lambda_perceptual"]})')

Model: dual | Parameters: 27,894,531
LR: 0.0002 | AMP: False
Loss: Charb(1.0) + Perc(0.0)


In [ ]:
# ── Cell 7: Full training loop ───────────────────────────────────────────────
training_log = run_training(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler, scaler,
    device, cfg,
    start_epoch=start_epoch,
    best_psnr=best_psnr,
)
print('Training complete.')

Epoch 1: 100%|██████████| 107/107 [01:03<00:00,  1.68it/s, loss=0.0294, gnorm=2.4]


Epoch 001/40 | Train 0.0451 | Val 0.0370 | PSNR 23.05 dB | SSIM 0.7321
  -> New best PSNR: 23.05 dB


Epoch 2: 100%|██████████| 107/107 [01:03<00:00,  1.69it/s, loss=0.0352, gnorm=2.6]


Epoch 002/40 | Train 0.0341 | Val 0.0367 | PSNR 23.26 dB | SSIM 0.7396
  -> New best PSNR: 23.26 dB


Epoch 3: 100%|██████████| 107/107 [01:03<00:00,  1.69it/s, loss=0.0405, gnorm=1.7]


...

Epoch 038/40 | Train 0.0205 | Val 0.0217 | PSNR 26.98 dB | SSIM 0.8542


Epoch 39: 100%|██████████| 107/107 [01:03<00:00,  1.68it/s, loss=0.0201, gnorm=0.1]


Epoch 039/40 | Train 0.0205 | Val 0.0215 | PSNR 26.96 dB | SSIM 0.8552


Epoch 40: 100%|██████████| 107/107 [01:03<00:00,  1.68it/s, loss=0.0184, gnorm=0.2]


Epoch 040/40 | Train 0.0207 | Val 0.0215 | PSNR 26.96 dB | SSIM 0.8570
Training complete.


In [ ]:
# ── Cell 8: Qualitative results ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from PIL import Image
from src.train import _forward

best_model = build_model(cfg, device)
best_model.load_state_dict(
    torch.load(os.path.join(cfg['checkpoint_dir'], 'best_model.pth'), map_location=device)
)
best_model.eval()

def load_vimeo_clip(clip_dir, patch_size=256):
    f0  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im1.png')).convert('RGB'))
    f1  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im3.png')).convert('RGB'))
    gt  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im2.png')).convert('RGB'))
    evt = torch.load(os.path.join(clip_dir, 'voxel.pt'), weights_only=True)
    if zero_events:
        evt = torch.zeros_like(evt)
    _, H, W = f0.shape
    top  = (H - patch_size) // 2
    left = (W - patch_size) // 2
    f0  = TF.crop(f0,  top, left, patch_size, patch_size)
    f1  = TF.crop(f1,  top, left, patch_size, patch_size)
    gt  = TF.crop(gt,  top, left, patch_size, patch_size)
    evt = evt[:, top:top+patch_size, left:left+patch_size]
    return f0, f1, gt, evt

show_dirs = val_dirs[:6]
fig, axes = plt.subplots(len(show_dirs), 5, figsize=(22, 4.5 * len(show_dirs)))
if len(show_dirs) == 1:
    axes = axes[np.newaxis, :]

for col, title in enumerate(['Frame f0', 'Frame f1', 'Ground Truth', 'Prediction', 'Error (x4)']):
    axes[0, col].set_title(title, fontsize=13, fontweight='bold')

with torch.no_grad():
    for row, clip_dir in enumerate(show_dirs):
        f0, f1, gt, evt = load_vimeo_clip(clip_dir)
        f0_d = f0.unsqueeze(0).to(device)
        f1_d = f1.unsqueeze(0).to(device)
        evt_d = evt.unsqueeze(0).to(device)
        outputs = _forward(best_model, f0_d, f1_d, evt_d)
        pred = outputs[0].squeeze(0).float().cpu().clamp(0, 1)

        error = (gt - pred).abs() * 4
        mse   = ((gt - pred) ** 2).mean().item()
        psnr  = -10 * np.log10(mse + 1e-10)

        for col, img in enumerate([f0, f1, gt, pred, error]):
            axes[row, col].imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
            axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        axes[row, 3].set_xlabel(f'PSNR: {psnr:.2f} dB', fontsize=9, color='steelblue')

title = f'{RUN_MODE} — Qualitative Results on Validation'
plt.suptitle(title, fontsize=15, y=1.01)
plt.tight_layout()
fig_path = os.path.join(cfg['checkpoint_dir'], 'qualitative_results.png')
plt.savefig(fig_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved to {fig_path}')

Output hidden; open in https://colab.research.google.com to view.